# Decision tree hyperparameters
max_depth on the Social Network Ads data, then every main hyperparameter on the two-moons data. The interactive version is `app.py`.

In [1]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from app import fit   # loads the data, splits it and trains a tree (same code as the app)

## max_depth on the Social Network Ads data

In [2]:
# Train one tree per depth; None means 'grow until every leaf is pure'
for depth in [None, 1, 2, 3, 4, 5]:
    tree, X_train, X_test, y_train, y_test, axes, classes = fit("ads", max_depth=depth)
    print(f"max_depth={depth}: {tree.get_n_leaves()} leaves, train {tree.score(X_train, y_train):.3f}, test {tree.score(X_test, y_test):.3f}")

max_depth=None: 49 leaves, train 1.000, test 0.910
max_depth=1: 2 leaves, train 0.820, test 0.890
max_depth=2: 4 leaves, train 0.910, test 0.940
max_depth=3: 8 leaves, train 0.917, test 0.940
max_depth=4: 13 leaves, train 0.933, test 0.930
max_depth=5: 20 leaves, train 0.943, test 0.950


In [3]:
# Choose max_depth honestly: 5-fold cross-validation on the training set only
grid = GridSearchCV(DecisionTreeClassifier(random_state=42),
                    {"max_depth": [1, 2, 3, 4, 5, 6, 8, 10, None]}, cv=5)
grid.fit(X_train, y_train)
print(grid.best_params_, round(grid.best_score_, 3), "test:", round(grid.score(X_test, y_test), 3))
# Average cross-validation accuracy for every depth
for d, s in zip(grid.cv_results_["param_max_depth"], grid.cv_results_["mean_test_score"]):
    print(f"max_depth={d}: {s:.3f}")

{'max_depth': 2} 0.9 test: 0.94
max_depth=1: 0.813
max_depth=2: 0.900
max_depth=3: 0.883
max_depth=4: 0.860
max_depth=5: 0.837
max_depth=6: 0.833
max_depth=8: 0.823
max_depth=10: 0.833
max_depth=None: 0.820


## Every hyperparameter on the two-moons data (375 training rows)

In [4]:
settings = [{}, {"criterion": "entropy"}, {"splitter": "random"}, {"max_depth": 1}, {"max_depth": 2}, {"max_depth": 5},
            {"min_samples_split": 100}, {"min_samples_split": 301}, {"min_samples_leaf": 20}, {"min_samples_leaf": 100},
            {"max_features": 1}, {"max_leaf_nodes": 5}, {"min_impurity_decrease": 0.01}, {"min_impurity_decrease": 0.1}]
for params in settings:
    tree, X_train, X_test, y_train, y_test, axes, classes = fit("moons", **params)
    print(f"{str(params):32s} depth {tree.get_depth():2d}, {tree.get_n_leaves():3d} leaves, "
          f"train {tree.score(X_train, y_train):.3f}, test {tree.score(X_test, y_test):.3f}")

{}                               depth 16,  43 leaves, train 1.000, test 0.856
{'criterion': 'entropy'}         depth 16,  44 leaves, train 1.000, test 0.832
{'splitter': 'random'}           depth 20, 102 leaves, train 1.000, test 0.864
{'max_depth': 1}                 depth  1,   2 leaves, train 0.819, test 0.824
{'max_depth': 2}                 depth  2,   4 leaves, train 0.899, test 0.896
{'max_depth': 5}                 depth  5,  13 leaves, train 0.936, test 0.896
{'min_samples_split': 100}       depth  4,   7 leaves, train 0.899, test 0.896
{'min_samples_split': 301}       depth  1,   2 leaves, train 0.819, test 0.824
{'min_samples_leaf': 20}         depth  5,  11 leaves, train 0.872, test 0.840
{'min_samples_leaf': 100}        depth  2,   3 leaves, train 0.819, test 0.824
{'max_features': 1}              depth 19,  50 leaves, train 1.000, test 0.864


{'max_leaf_nodes': 5}            depth  3,   5 leaves, train 0.899, test 0.896


{'min_impurity_decrease': 0.01}  depth  5,   7 leaves, train 0.933, test 0.904
{'min_impurity_decrease': 0.1}   depth  1,   2 leaves, train 0.819, test 0.824


In [5]:
# min_samples_split = 301: the root (375 rows) splits once, its children are too small to split again
tree, *_ = fit("moons", min_samples_split=301)
print(export_text(tree, feature_names=["x1", "x2"], show_weights=True))

|--- x2 <= 0.38
|   |--- weights: [49.00, 167.00] class: 1
|--- x2 >  0.38
|   |--- weights: [140.00, 19.00] class: 0



In [6]:
# min_impurity_decrease: the weighted decrease of the root split, as scikit-learn computes it
tree, *_ = fit("moons")
t = tree.tree_
N = t.n_node_samples[0]
L, R = t.children_left[0], t.children_right[0]
dec = t.n_node_samples[0] / N * (t.impurity[0]
       - t.n_node_samples[L] / t.n_node_samples[0] * t.impurity[L]
       - t.n_node_samples[R] / t.n_node_samples[0] * t.impurity[R])
print("root Gini", round(t.impurity[0], 3), "children", round(t.impurity[L], 3), round(t.impurity[R], 3),
      "sizes", t.n_node_samples[L], t.n_node_samples[R], "decrease", round(dec, 3))

root Gini 0.5 children 0.351 0.21 sizes 216 159 decrease 0.209


## Randomness knobs on a single tree: average over 50 moons datasets
One split hides the effect, so we redraw the data 50 times and average test accuracy.

In [7]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
S = {"best (default)": {}, "entropy": {"criterion": "entropy"}, "splitter=random": {"splitter": "random"}, "max_features=1": {"max_features": 1}}
acc, leaves = {k: [] for k in S}, {k: [] for k in S}
for seed in range(50):
    X, y = make_moons(n_samples=500, noise=0.3, random_state=seed)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=seed)
    for k, p in S.items():
        m = DecisionTreeClassifier(random_state=seed, **p).fit(X_tr, y_tr)
        acc[k].append(m.score(X_te, y_te)); leaves[k].append(m.get_n_leaves())
for k in S:
    print(f"{k:16s} test {np.mean(acc[k]):.3f}, leaves {np.mean(leaves[k]):.0f}")

best (default)   test 0.866, leaves 45
entropy          test 0.867, leaves 44
splitter=random  test 0.862, leaves 99
max_features=1   test 0.857, leaves 54


## Cost-complexity pruning (ccp_alpha) on the Social Network Ads data
Grow the full tree, then cut it back. `cost_complexity_pruning_path` lists the alphas at which the best subtree changes.

In [ ]:
from sklearn.model_selection import cross_val_score
tree, X_train, X_test, y_train, y_test, axes, classes = fit("ads")
path = tree.cost_complexity_pruning_path(X_train, y_train)
rows = []
for alpha, impurity in zip(path.ccp_alphas, path.impurities):
    pruned = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha).fit(X_train, y_train)
    cv = cross_val_score(DecisionTreeClassifier(random_state=42, ccp_alpha=alpha), X_train, y_train, cv=5).mean()
    rows.append((alpha, impurity, pruned.get_n_leaves(), cv, pruned.score(X_test, y_test)))
    print(f"alpha {alpha:.5f}: leaf impurity {impurity:.4f}, {rows[-1][2]} leaves, CV {cv:.3f}, test {rows[-1][4]:.3f}")
best = max(rows, key=lambda r: r[3])
print("cross-validation picks alpha", round(best[0], 5), "->", best[2], "leaves")

In [ ]:
# Tree score = total leaf impurity + alpha * number of leaves, for four subtrees of the path
subtrees = {r[2]: r[1] for r in rows if r[2] in (49, 3, 2, 1)}
for alpha in [0, 0.01, 0.2]:
    scores = {leaves: round(imp + alpha * leaves, 3) for leaves, imp in subtrees.items()}
    print(f"alpha {alpha}: {scores} -> lowest: {min(scores, key=scores.get)} leaves")